# 10 — Metrics / Observability Pattern

Always produce metrics from pipeline.

Process:

INPUT
  |
  v
PROCESS
  |
  +--> DATA OUTPUT
  +--> METRICS OUTPUT

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("metrics-pattern")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

## Step 1 — Input

In [2]:
df = spark.createDataFrame(
    [
        ("e1","u1",10.0),
        ("e2",None,20.0),
        ("e3","u2",-5.0),
    ],
    ["event_id","user_id","amount"]
)

df.show()

+--------+-------+------+
|event_id|user_id|amount|
+--------+-------+------+
|      e1|     u1|  10.0|
|      e2|   NULL|  20.0|
|      e3|     u2|  -5.0|
+--------+-------+------+



## Step 2 — Process

In [3]:
processed = (
    df
    .withColumn("is_valid",
        (~F.col("user_id").isNull()) & (F.col("amount") >= 0)
    )
)

## Step 3 — Outputs

In [4]:
valid = processed.filter("is_valid")
invalid = processed.filter("NOT is_valid")

## Step 4 — Metrics

In [5]:
metrics = spark.createDataFrame(
    [
        ("rows_in", df.count()),
        ("rows_valid", valid.count()),
        ("rows_invalid", invalid.count()),
    ],
    ["metric","value"]
)

metrics.show()

+------------+-----+
|      metric|value|
+------------+-----+
|     rows_in|    3|
|  rows_valid|    1|
|rows_invalid|    2|
+------------+-----+



## Step 5 — Combine pattern

In [6]:
def run(df):
    processed = (
        df
        .withColumn("is_valid",
            (~F.col("user_id").isNull()) & (F.col("amount") >= 0)
        )
    )

    valid = processed.filter("is_valid")
    invalid = processed.filter("NOT is_valid")

    metrics = spark.createDataFrame(
        [
            ("rows_in", df.count()),
            ("rows_valid", valid.count()),
            ("rows_invalid", invalid.count()),
        ],
        ["metric","value"]
    )

    return {
        "data": valid,
        "quarantine": invalid,
        "metrics": metrics
    }

result = run(df)
result["metrics"].show()

+------------+-----+
|      metric|value|
+------------+-----+
|     rows_in|    3|
|  rows_valid|    1|
|rows_invalid|    2|
+------------+-----+



In [7]:
spark.stop()